# Phase 5 — Feature Engineering & Spatial Integration (GeoPandas)

## 1. System Architecture & Objectives
This notebook executes feature engineering and spatial enrichment to address the primary failure mode of our physical-only baselines: the complete absence of location dynamics and non-linear structural interaction terms. 

### Core Objectives
* **Ingest Cleaned Datasets**: Load `train_cleaned.parquet` and `test_cleaned.parquet` from `artifacts/` to recover preserved coordinates (`Latitude`, `Longitude`) and secondary attributes (`City`, `PostalCode`, amenities).
* **Structural Domain Feature Engineering**: Formulate hedonic ratio features:
  * `PropertyAge` and `EffectiveAge` from `YearBuilt`.
  * `BedBathRatio` (`BedroomsTotal / BathroomsTotalInteger`).
  * `LivingAreaRatio` (`LivingArea / LotSizeSquareFeet`).
  * Binary flags for amenity interactions (`HasPool`, `HasFireplace`).
* **Spatial Polygon Integration (GeoPandas)**: 
  * Convert coordinate pairs (`Latitude`, `Longitude`) into EPSG:4326 Shapely Point geometries.
  * Ingest the California School District boundary shapefile / GeoJSON.
  * Execute a spatial point-in-polygon join (`gpd.sjoin`) to assign every property to its Unified/Unified High School District catchment area without coordinate leakage.
* **Leakage-Safe High-Cardinality Encoding**: Apply Target Encoding (out-of-fold / smoothing) or Frequency Encoding on spatial boundaries (`SchoolDistrict`, `PostalCode`) strictly fit on the training partition.
* **Artifact Export**: Export the enriched feature matrices (`X_train_engineered.parquet`, `X_test_engineered.parquet`) ready for gradient boosting and ensemble retraining.

In [8]:
import os
from pathlib import Path
import joblib
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print("GeoPandas and engineering dependencies initialized.")

GeoPandas and engineering dependencies initialized.


## 2. Ingest Partitioned Clean Datasets

We retrieve the master parquet partitions generated during the data preparation phase. These tables preserve coordinates, dates, and candidate metadata that were withheld from the initial baseline model.

In [10]:
artifacts_dir = Path("artifacts")

# Ingest master cleaned splits
train_df = pd.read_parquet(artifacts_dir / "train_cleaned.parquet")
test_df = pd.read_parquet(artifacts_dir / "test_cleaned.parquet")

print(f"Loaded train_df: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
print(f"Loaded test_df : {test_df.shape[0]:,} rows × {test_df.shape[1]} columns")
assert "Latitude" in train_df.columns and "Longitude" in train_df.columns, "Coordinates missing!"

Loaded train_df: 293,758 rows × 18 columns
Loaded test_df : 12,336 rows × 18 columns


## 3. Structural Domain Feature Engineering

We generate structural domain ratios based on appraisal principles:
1. **`PropertyAge`**: Linear age computed relative to transaction year:
   $$\text{PropertyAge} = \text{CloseDate.dt.year} - \text{YearBuilt}$$
2. **`BedBathRatio`**: Captures functional layout utility. Homes with disproportionate bedroom counts relative to bathrooms incur economic utility penalties.
3. **`LivingToLotRatio`**: Ratio of structural footprint to total parcel area:
   $$\text{LivingToLotRatio} = \frac{\text{LivingArea}}{\text{LotSizeSquareFeet}}$$
4. **Amenity Flags**: Parse boolean or categorical fields (`PoolPrivateYN`, `FireplaceYN`) into clean binary indicators ($0/1$).

In [11]:
def engineer_structural_features(df):
    """Computes appraisal-informed structural ratios and clean flags."""
    df_out = df.copy()
    
    # 1. Age of property at time of sale
    sale_year = df_out["CloseDate"].dt.year if "CloseDate" in df_out.columns else 2026
    df_out["PropertyAge"] = np.maximum(0, sale_year - df_out["YearBuilt"])
    
    # 2. Bed-to-Bath Utility Ratio
    # Add epsilon to denominator to prevent division by zero
    df_out["BedBathRatio"] = df_out["BedroomsTotal"] / np.maximum(1, df_out["BathroomsTotalInteger"])
    
    # 3. Living Area to Lot Size Ratio
    df_out["LivingToLotRatio"] = df_out["LivingArea"] / np.maximum(df_out["LivingArea"], df_out["LotSizeSquareFeet"])
    
    # 4. Binary Amenity Encodings
    df_out["HasPool"] = df_out["PoolPrivateYN"].astype(str).str.upper().isin(["TRUE", "1", "Y", "YES"]).astype(int)
    df_out["HasFireplace"] = df_out["FireplaceYN"].astype(str).str.upper().isin(["TRUE", "1", "Y", "YES"]).astype(int)
    
    return df_out

train_df = engineer_structural_features(train_df)
test_df = engineer_structural_features(test_df)

print("Structural feature engineering complete.")
display(train_df[["PropertyAge", "BedBathRatio", "LivingToLotRatio", "HasPool", "HasFireplace"]].head())

Structural feature engineering complete.


,PropertyAge,BedBathRatio,LivingToLotRatio,HasPool,HasFireplace
0,39.0000,1.5000,0.1587,0,1
1,16.0000,1.3333,0.3672,0,0
2,39.0000,1.3333,0.4177,0,1
3,72.0000,3.0000,0.2009,1,0
4,54.0000,1.5000,0.1360,0,0


## 4. Spatial Geometry Conversion & School District Point-in-Polygon Join

Real estate valuations are strongly bounded by school district catchment borders, which dictate public school access and local property tax rates. 

### Implementation Flow
1. Convert latitude and longitude fields into a GeoPandas `GeoDataFrame` using the WGS84 coordinate reference system (`EPSG:4326`).
2. Load California School District boundary polygons (or county/city geographic boundaries if school boundaries are local).
3. Align CRS coordinates to ensure matching planar projections.
4. Execute `gpd.sjoin(..., how="left", predicate="within")` to assign each listing to its geographic administrative district.

In [12]:
# Convert DataFrames to GeoDataFrames
gdf_train = gpd.GeoDataFrame(
    train_df,
    geometry=gpd.points_from_xy(train_df["Longitude"], train_df["Latitude"]),
    crs="EPSG:4326"
)

gdf_test = gpd.GeoDataFrame(
    test_df,
    geometry=gpd.points_from_xy(test_df["Longitude"], test_df["Latitude"]),
    crs="EPSG:4326"
)

print(f"GeoDataFrames instantiated with CRS: {gdf_train.crs}")

# Look for school district spatial boundaries
# Standard download path: data/shapefiles/ca_school_districts.geojson (or fallback to PostalCode/City clustering if boundary file not present)
boundary_paths = [
    Path("data/ca_school_districts.geojson"),
    Path("data/ca_school_districts.shp"),
    Path("ca_school_districts.geojson")
]

boundary_file = next((p for p in boundary_paths if p.exists()), None)

if boundary_file:
    print(f"Loading spatial polygons from {boundary_file}...")
    districts_gdf = gpd.read_file(boundary_file)
    if districts_gdf.crs != "EPSG:4326":
        districts_gdf = districts_gdf.to_crs("EPSG:4326")
        
    district_col_name = "NAME" if "NAME" in districts_gdf.columns else districts_gdf.columns[0]
    
    # Point-in-polygon spatial join
    gdf_train = gpd.sjoin(gdf_train, districts_gdf[[district_col_name, "geometry"]], how="left", predicate="within")
    gdf_train.rename(columns={district_col_name: "SchoolDistrict"}, inplace=True)
    gdf_train.drop(columns=["index_right"], inplace=True, errors="ignore")
    
    gdf_test = gpd.sjoin(gdf_test, districts_gdf[[district_col_name, "geometry"]], how="left", predicate="within")
    gdf_test.rename(columns={district_col_name: "SchoolDistrict"}, inplace=True)
    gdf_test.drop(columns=["index_right"], inplace=True, errors="ignore")
    
    print(f"Spatial join successful. Assigned {gdf_train['SchoolDistrict'].nunique():,} unique school districts.")
else:
    print("Dedicated district polygon file not located locally. Constructing spatial cluster proxies from PostalCode and City coordinates.")
    gdf_train["SchoolDistrict"] = gdf_train["PostalCode"].astype(str).str.zfill(5)
    gdf_test["SchoolDistrict"] = gdf_test["PostalCode"].astype(str).str.zfill(5)

# Convert back to standard DataFrames with spatial fields preserved
train_df = pd.DataFrame(gdf_train.drop(columns=["geometry"], errors="ignore"))
test_df = pd.DataFrame(gdf_test.drop(columns=["geometry"], errors="ignore"))

# Impute unassigned districts with 'Unknown'
train_df["SchoolDistrict"] = train_df["SchoolDistrict"].fillna("Unknown")
test_df["SchoolDistrict"] = test_df["SchoolDistrict"].fillna("Unknown")

GeoDataFrames instantiated with CRS: EPSG:4326
Dedicated district polygon file not located locally. Constructing spatial cluster proxies from PostalCode and City coordinates.


## 5. Leakage-Safe Target Encoding of Spatial Boundaries

High-cardinality geographic labels (`SchoolDistrict`, `PostalCode`, `City`) contain hundreds of unique values. One-hot encoding creates extreme matrix sparsity, while simple label encoding imposes arbitrary order.

We apply **Target Encoding with Empirical Bayes Smoothing**:
$$S_i = \lambda_i \bar{y}_i + (1 - \lambda_i) \bar{y}_{\text{global}}$$
$$\lambda_i = \frac{n_i}{n_i + m}$$

Where:
* $\bar{y}_i$ is the average sale price inside district $i$.
* $n_i$ is the transaction count in district $i$.
* $m$ is the smoothing weight (smoothing factor = 20) to pull small-sample estimates toward the global mean $\bar{y}_{\text{global}}$.
* **Leakage Guard**: The mapping is computed **strictly on `train_df`** and mapped directly onto `test_df`. Any district in `test_df` not present in training receives the global mean.

In [13]:
def fit_target_encoder(df_train, cat_col, target_col, smoothing_weight=20):
    """Fits an empirical Bayes smoothed target encoding dictionary."""
    global_mean = df_train[target_col].mean()
    stats = df_train.groupby(cat_col)[target_col].agg(["count", "mean"])
    
    # Smoothing formula
    smoothing = stats["count"] / (stats["count"] + smoothing_weight)
    smoothed_values = smoothing * stats["mean"] + (1 - smoothing) * global_mean
    
    mapping_dict = smoothed_values.to_dict()
    return mapping_dict, global_mean

# Encode SchoolDistrict
district_map, global_price_mean = fit_target_encoder(train_df, "SchoolDistrict", "ClosePrice", smoothing_weight=20)
train_df["District_TargetEnc"] = train_df["SchoolDistrict"].map(district_map).fillna(global_price_mean)
test_df["District_TargetEnc"] = test_df["SchoolDistrict"].map(district_map).fillna(global_price_mean)

# Encode PostalCode
postal_map, _ = fit_target_encoder(train_df, "PostalCode", "ClosePrice", smoothing_weight=25)
train_df["PostalCode_TargetEnc"] = train_df["PostalCode"].map(postal_map).fillna(global_price_mean)
test_df["PostalCode_TargetEnc"] = test_df["PostalCode"].map(postal_map).fillna(global_price_mean)

print(f"Target encodings calculated successfully. Global mean price: ${global_price_mean:,.2f}")
display(train_df[["SchoolDistrict", "District_TargetEnc", "PostalCode", "PostalCode_TargetEnc"]].head())

Target encodings calculated successfully. Global mean price: $1,195,391.86


,SchoolDistrict,District_TargetEnc,PostalCode,PostalCode_TargetEnc
0,92376,587163.7039,92376,591508.1908
1,93552,534514.4014,93552,541635.9257
2,93021,1195077.9006,93021,1195081.0402
3,92504,774256.1129,92504,777234.4421
4,93536,566614.4113,93536,568924.3946


## 6. Spatial Distance to Major Economic Anchors

Properties in California price along commuter access corridors to primary economic hubs (e.g., Downtown Los Angeles, Silicon Valley / San Jose, San Francisco). We calculate the Haversine distance from each property's coordinates to regional reference centroids to introduce continuous spatial decay surfaces.

In [14]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    """Vectorized calculation of spherical distance in miles."""
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Centroids for major economic cores
HUBS = {
    "Dist_DTLA": (34.0522, -118.2437),
    "Dist_SiliconValley": (37.3382, -121.8863),
    "Dist_SanDiego": (32.7157, -117.1611)
}

for hub_name, (h_lat, h_lon) in HUBS.items():
    train_df[hub_name] = haversine_vectorized(train_df["Latitude"], train_df["Longitude"], h_lat, h_lon)
    test_df[hub_name] = haversine_vectorized(test_df["Latitude"], test_df["Longitude"], h_lat, h_lon)

print("Economic anchor distance features computed:")
display(train_df[["Latitude", "Longitude", "Dist_DTLA", "Dist_SiliconValley", "Dist_SanDiego"]].head())

Economic anchor distance features computed:


,Latitude,Longitude,Dist_DTLA,Dist_SiliconValley,Dist_SanDiego
0,34.0886,-117.3760,49.7261,338.2049,95.6663
1,34.5553,-118.0243,36.9473,289.1516,136.4596
2,34.2710,-118.8948,40.1755,270.1769,146.7143
3,33.9651,-117.4067,48.3158,342.8039,87.4814
4,34.6641,-118.2005,42.3495,276.6968,147.2844


## 7. Compile Final Enriched Feature Matrices

We assemble the consolidated feature set combining:
1. **Core Physical Drivers**: `LivingArea`, `BedroomsTotal`, `BathroomsTotalInteger`, `LotSizeSquareFeet`, `YearBuilt`, `GarageSpaces`, `Stories`.
2. **Engineered Ratios & Amenities**: `PropertyAge`, `BedBathRatio`, `LivingToLotRatio`, `HasPool`, `HasFireplace`.
3. **Spatial Indicators**: `Latitude`, `Longitude`, `District_TargetEnc`, `PostalCode_TargetEnc`, `Dist_DTLA`, `Dist_SiliconValley`, `Dist_SanDiego`.

In [15]:
ENGINEERED_FEATURE_COLUMNS = [
    # Physical Drivers
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "Stories",
    # Engineered Ratios & Flags
    "PropertyAge",
    "BedBathRatio",
    "LivingToLotRatio",
    "HasPool",
    "HasFireplace",
    # Geographic & Spatial
    "Latitude",
    "Longitude",
    "District_TargetEnc",
    "PostalCode_TargetEnc",
    "Dist_DTLA",
    "Dist_SiliconValley",
    "Dist_SanDiego"
]

TARGET = "ClosePrice"

# Impute any missing numerical features with training medians strictly fit on train
for col in ENGINEERED_FEATURE_COLUMNS:
    if train_df[col].isnull().any() or test_df[col].isnull().any():
        med_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(med_val)
        test_df[col] = test_df[col].fillna(med_val)

X_train_eng = train_df[ENGINEERED_FEATURE_COLUMNS].copy()
y_train = train_df[TARGET].values

X_test_eng = test_df[ENGINEERED_FEATURE_COLUMNS].copy()
y_test = test_df[TARGET].values

print(f"X_train_eng final shape: {X_train_eng.shape}")
print(f"X_test_eng  final shape: {X_test_eng.shape}")
assert not X_train_eng.isnull().any().any(), "Null values remain in X_train_eng!"
assert not X_test_eng.isnull().any().any(), "Null values remain in X_test_eng!"

X_train_eng final shape: (293758, 19)
X_test_eng  final shape: (12336, 19)


## 8. Export Engineered Artifacts

We save the engineered parquet tables and updated column schemas to `artifacts/` for gradient boosted model training (LightGBM / XGBoost) in the next phase.

In [16]:
# 1. Export Parquet feature matrices
X_train_eng.to_parquet(artifacts_dir / "X_train_engineered.parquet", index=False)
X_test_eng.to_parquet(artifacts_dir / "X_test_engineered.parquet", index=False)

# 2. Save targets
np.save(artifacts_dir / "y_train_engineered.npy", y_train)
np.save(artifacts_dir / "y_test_engineered.npy", y_test)

# 3. Export feature names
joblib.dump(ENGINEERED_FEATURE_COLUMNS, artifacts_dir / "engineered_feature_names.joblib")

print("--- Phase 5 Artifacts Successfully Persisted ---")
print(f" - {artifacts_dir / 'X_train_engineered.parquet'}")
print(f" - {artifacts_dir / 'X_test_engineered.parquet'}")
print(f" - {artifacts_dir / 'y_train_engineered.npy'}")
print(f" - {artifacts_dir / 'y_test_engineered.npy'}")
print(f" - {artifacts_dir / 'engineered_feature_names.joblib'}")

--- Phase 5 Artifacts Successfully Persisted ---
 - artifacts/X_train_engineered.parquet
 - artifacts/X_test_engineered.parquet
 - artifacts/y_train_engineered.npy
 - artifacts/y_test_engineered.npy
 - artifacts/engineered_feature_names.joblib
